# Experimental test 8

    test ('vq',              # llm_model
        'models--cyankiwi--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',         # model_ver
        1,                # few_shot_n
        1,                # test_n(# of question for test)
        'Y',              # q_src_yn 
        1,                # iteration num
        'sys_prompt10',   # prompt ver
        1,                # self-consistency number
        0.01,             # temperature
        'ver7'            # excel_verion
        )


In [1]:
import os
import pandas as pd
from config import config as conf
import re
import numpy as np
from sklearn import metrics



In [2]:
def sc_calc_acc_condition_with_temp_with_sc(llm_model, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')] 
    print(f'len of opt_file : {len(opt_file)}')

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            # print(f'size of the dataset : {df_eval.shape[0]}')
            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list


In [ ]:
def sc_calc_acc_condition_with_temp_with_sc_model(llm_model, model_ver, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}/{model_ver}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')] 
    print(f'len of opt_file : {len(opt_file)}')

    df = pd.DataFrame()
    save_df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]
            print(tmp.shape)
            save_df = pd.concat([save_df, tmp], axis=0, ignore_index=True)

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            # print(f'size of the dataset : {df_eval.shape[0]}')
            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list, save_df


In [19]:
#    test ('vq',              # llm_model
#         'models--cyankiwi--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',         # model_ver
#         3,                # few_shot_n
#         30,                # test_n(# of question for test)
#         'Y',              # q_src_yn 
#         10,                # iteration num
#         'sys_prompt10',   # prompt ver
#         5,                # self-consistency number
#         0.01,             # temperature
#         'ver7'            # excel_verion
#         )

In [20]:
list_, df_=         sc_calc_acc_condition_with_temp_with_sc_model('vq', 'models--cyankiwi--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',  3, 30, 'Y', 10, 'sys_prompt10', 5,  0.01, 'ver7')
# print(list_)

len of opt_file : 10
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
              precision    recall  f1-score   support

           0      1.000     0.920     0.958        75
           1      0.769     0.964     0.856        83
           2      0.900     0.600     0.720        45

    accuracy                          0.867       203
   macro avg      0.890     0.828     0.845       203
weighted avg      0.883     0.867     0.864       203

vq_result_3_30_Y :  86.69950738916256


In [21]:
list_, df_ =         sc_calc_acc_condition_with_temp_with_sc_model('vq', 'models--cyankiwi--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',  4, 30, 'Y', 10, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

len of opt_file : 10
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
(150, 6)
              precision    recall  f1-score   support

           0      0.985     0.892     0.936        74
           1      0.804     0.947     0.870        95
           2      0.895     0.708     0.791        48

    accuracy                          0.876       217
   macro avg      0.894     0.849     0.865       217
weighted avg      0.886     0.876     0.875       217

vq_result_4_30_Y :  87.55760368663594
[np.float64(88.0), np.float64(91.30434782608695), np.float64(84.21052631578947), np.float64(100.0), np.float64(82.6086956521739), np.float64(81.81818181818183), np.float64(75.0), np.float64(87.5), np.float64(89.47368421052632), np.float64(95.0)]


In [22]:
df_.to_csv('chk.csv')